In [1]:
import numpy as np
import fiftyone as fo
import fiftyone.utils.random as four
import fiftyone.zoo as foz
import cv2
import pyautogui
import os
import time
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict, Counter
import pandas as pd
from ultralytics import YOLO

In [2]:
def load_data():
    dataset = (
        foz.load_zoo_dataset("quickstart")
        .select_fields("ground_truth")
        .set_field("tags", [])
    ).clone()
    
    #Split
    four.random_split(dataset, {"train": 0.7, "test": 0.2, "val": 0.1}, seed=4)
    return dataset

    
def export():#Export
    for split in splits:
        split_view = dataset.match_tags(split)
        split_view.export(
            export_dir=export_dir,
            dataset_type=fo.types.YOLOv5Dataset,
            label_field=label_field,
            split=split,
            classes=classes,
        )


def save_image(label, annotated_image):
    cv2.imwrite(f"{label}_annotated_best.jpg", annotated_image)
    print(f"\nAnnotated image saved as '{label}_annotated_best.jpg'")


def model_test(dataset):
    correct_predictions = 0
    total_predictions = 0
    total_samples = 0
    
    for sample in dataset.match_tags("test"):
        print(f"Processing: {sample.filepath}")
        
        # Run YOLO prediction
        results = model.predict(sample.filepath, conf=0.5, verbose=False)
        
        # Get ground truth labels
        ground_truth_labels = set()
        if sample.ground_truth is not None:
            for detection in sample.ground_truth.detections:
                ground_truth_labels.add(detection.label)
        
        # Get predicted labels
        predicted_labels = set()
        if len(results) > 0 and results[0].boxes is not None:
            for box in results[0].boxes:
                if box.cls is not None:
                    # Convert class index to label name - fixed deprecation warning
                    class_idx = int(box.cls.cpu().numpy().item())
                    label_name = results[0].names[class_idx]
                    predicted_labels.add(label_name)
        
        # Calculate accuracy for this sample
        if len(ground_truth_labels) > 0:
            # Check how many ground truth labels were correctly predicted
            correct_in_sample = len(ground_truth_labels.intersection(predicted_labels))
            correct_predictions += correct_in_sample
            total_predictions += len(ground_truth_labels)
            
            print(f"  Ground truth: {ground_truth_labels}")
            print(f"  Predicted: {predicted_labels}")
            print(f"  Correct: {correct_in_sample}/{len(ground_truth_labels)}")
        
        total_samples += 1
        print("---")
    
    # Calculate overall accuracy
    if total_predictions > 0:
        accuracy = (correct_predictions / total_predictions) * 100
        print(f"\nOverall Accuracy: {accuracy:.2f}% ({correct_predictions}/{total_predictions})")
        print(f"Total samples processed: {total_samples}")
    else:
        print("No ground truth labels found in test set")


def show_image(results):
    annotated_image = results[0].plot()

    # Count detected objects
    counts = Counter()
    if results[0].boxes is not None:
        for box in results[0].boxes:
            class_idx = int(box.cls.item())
            label_name = results[0].names[class_idx]
            counts[label_name] += 1
    
    # Parameters for scaling overlay based on image size
    img_height, img_width, _ = annotated_image.shape
    overlay_width = int(img_width * 0.3)  # 30% of image width
    font_scale = img_width / 1000  # Scale font based on image width
    font_thickness = max(1, int(img_width / 500))  # Scale thickness
    padding = int(img_width / 100)  # Padding scales with image
    
    # Create overlay content
    overlay_lines = []
    if counts:
        for obj_label, count in counts.items():
            overlay_lines.append(f"{obj_label}: {count}")
    else:
        overlay_lines.append("No detections")
    
    # Calculate overlay height based on content
    line_height = int(30 * font_scale)
    overlay_height = int((len(overlay_lines) + 0.5) * line_height)
    
    # Create semi-transparent background
    overlay = annotated_image.copy()
    # Fix: Ensure coordinates are properly formatted as integer tuples
    cv2.rectangle(
        overlay, 
        (int(padding), int(padding)), 
        (int(padding + overlay_width), int(padding + overlay_height)), 
        (0, 0, 0), 
        -1
    )
    annotated_image = cv2.addWeighted(overlay, 0.6, annotated_image, 0.4, 0)  # 60% opacity
    
    # Add text to the overlay
    for i, line in enumerate(overlay_lines):
        y_pos = padding + (i + 1) * line_height
        cv2.putText(
            annotated_image, 
            line, 
            (padding * 2, y_pos), 
            cv2.FONT_HERSHEY_SIMPLEX, 
            font_scale, 
            (255, 255, 255), 
            font_thickness, 
            cv2.LINE_AA
        )
    
    # 1. Get the original dimensions of the annotated image.
    # We'll use the shape of the image array (height, width, channels)
    original_height, original_width, _ = annotated_image.shape
    
    # 2. Get the screen resolution
    screen_width, screen_height = pyautogui.size()
    
    # 3. Calculate the aspect ratio
    aspect_ratio = original_width / original_height
    
    # 4. Calculate new dimensions that maintain the aspect ratio
    if (screen_width / screen_height) > aspect_ratio:
        # Screen is wider, so scale by height
        new_height = screen_height
        new_width = int(new_height * aspect_ratio)
    else:
        # Screen is taller, so scale by width
        new_width = screen_width
        new_height = int(new_width / aspect_ratio)
    
    # 5. Resize the image with the new dimensions, then scale down
    resized_image = cv2.resize(annotated_image, (round(new_width*.8), round(new_height*.8)))
    
    # 6. Display the resized image
    cv2.imshow("YOLO Detections", resized_image)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

    return annotated_image


def process_video(input_path, output_path=None, conf=0.5):
    """
    Process video with YOLO detection and overlay object counts with elapsed time
    
    Args:
        input_path: Path to input video file or camera index (0 for webcam)
        output_path: Optional path to save output video
        conf: Confidence threshold for YOLO detection
    """
    # Input can be file path or camera index
    if isinstance(input_path, int) or (isinstance(input_path, str) and input_path.isdigit()):
        cap = cv2.VideoCapture(int(input_path))
        source_name = f"Camera {input_path}"
        is_webcam = True
    else:
        cap = cv2.VideoCapture(input_path)
        source_name = input_path
        is_webcam = False
    
    if not cap.isOpened():
        print(f"Error: Could not open {source_name}")
        return
    
    # Get video properties
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    
    # Initialize video writer if output path is provided
    writer = None
    if output_path:
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Codec (can change to 'XVID' if needed)
        writer = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))
    
    print(f"Processing {source_name}...")
    print(f"Video FPS: {fps}")
    print("Press 'q' to quit")
    
    frame_count = 0
    start_time = time.time()  # For webcam real-time processing
    elapsed_seconds = 0
    
    try:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            
            frame_count += 1
            
            # Calculate elapsed video time (in seconds)
            if is_webcam:
                # For webcam, use actual elapsed processing time
                elapsed_seconds = time.time() - start_time
            else:
                # For video files, calculate based on frame count and fps
                elapsed_seconds = frame_count / fps
            
            # Format time as MM:SS.ms
            minutes = int(elapsed_seconds // 60)
            seconds = int(elapsed_seconds % 60)
            milliseconds = int((elapsed_seconds % 1) * 1000)
            time_str = f"{minutes:02d}:{seconds:02d}.{milliseconds:03d}"
            
            # Run YOLO prediction on the frame
            results = model.predict(frame, conf=conf, verbose=False)
            
            # Plot detections on the frame
            annotated_frame = results[0].plot()
            
            # Count detected objects
            counts = Counter()
            if results[0].boxes is not None:
                for box in results[0].boxes:
                    class_idx = int(box.cls.item())
                    label_name = results[0].names[class_idx]
                    counts[label_name] += 1
            
            # Parameters for scaling overlay based on frame size
            img_height, img_width, _ = annotated_frame.shape
            overlay_width = int(img_width * 0.3)  # 30% of frame width
            font_scale = max(0.5, img_width / 1000)  # Scale font based on frame width
            font_thickness = max(1, int(img_width / 500))  # Scale thickness
            padding = int(img_width / 100)  # Padding scales with frame
            
            # Create overlay content
            overlay_lines = []
            if counts:
                for obj_label, count in counts.items():
                    overlay_lines.append(f"{obj_label}: {count}")
            else:
                overlay_lines.append("No detections")
            
            # Calculate overlay height based on content
            line_height = int(30 * font_scale)
            overlay_height = int((len(overlay_lines) + 0.5) * line_height)
            
            # Create semi-transparent background for object counts
            overlay = annotated_frame.copy()
            cv2.rectangle(
                overlay, 
                (int(padding), int(padding)), 
                (int(padding + overlay_width), int(padding + overlay_height)), 
                (0, 0, 0), 
                -1
            )
            # Apply the semi-transparent overlay
            annotated_frame = cv2.addWeighted(overlay, 0.6, annotated_frame, 0.4, 0)
            
            # Add text to the overlay
            for i, line in enumerate(overlay_lines):
                y_pos = int(padding + (i + 1) * line_height)
                cv2.putText(
                    annotated_frame, 
                    line, 
                    (int(padding * 2), y_pos), 
                    cv2.FONT_HERSHEY_SIMPLEX, 
                    font_scale, 
                    (255, 255, 255), 
                    font_thickness, 
                    cv2.LINE_AA
                )
            
            # Add frame counter and elapsed time in bottom right
            time_info = f"Frame: {frame_count} | Time: {time_str}"
            text_size = cv2.getTextSize(time_info, cv2.FONT_HERSHEY_SIMPLEX, font_scale, font_thickness)[0]
            
            # Create semi-transparent background for time display
            time_overlay = annotated_frame.copy()
            cv2.rectangle(
                time_overlay,
                (img_width - text_size[0] - padding*3, img_height - text_size[1] - padding*3),
                (img_width - padding, img_height - padding),
                (0, 0, 0),
                -1
            )
            # Apply the semi-transparent overlay for time display - FIXED THIS PART
            annotated_frame = cv2.addWeighted(time_overlay, 0.6, annotated_frame, 0.4, 0)
            
            # Add text on semi-transparent background
            cv2.putText(
                annotated_frame,
                time_info,
                (img_width - text_size[0] - padding*2, img_height - padding*2),
                cv2.FONT_HERSHEY_SIMPLEX,
                font_scale,
                (255, 255, 255),
                font_thickness,
                cv2.LINE_AA
            )
            
            # Display the frame with overlay
            cv2.imshow("YOLO Video Detections", annotated_frame)
            
            # Write frame to output video if enabled
            if writer:
                writer.write(annotated_frame)
            
            # Exit on 'q' key press
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
    
    except Exception as e:
        print(f"Error during processing: {e}")
        import traceback
        traceback.print_exc()
    
    finally:
        # Release resources
        cap.release()
        if writer:
            writer.release()
        cv2.destroyAllWindows()
        print(f"Video processing complete. Processed {frame_count} frames in {elapsed_seconds:.2f} seconds.")

# Example usage:
# process_video("input_video.mp4", "output_video.mp4")
# process_video(0)  # For webcam

In [4]:
dataset = load_data()

Dataset already downloaded
You are running the oldest supported major version of MongoDB. Please refer to https://deprecation.voxel51.com for deprecation notices. You can suppress this exception by setting your `database_validation` config parameter to `False`. See https://docs.voxel51.com/user_guide/config.html#configuring-a-mongodb-connection for more information
Loading existing dataset 'quickstart'. To reload from disk, either delete the existing dataset or provide a custom `dataset_name` to use


In [5]:
splits = ["train","test","val"]
label_field = "ground_truth"
export_dir = "yolov5_dataset"
classes = dataset.distinct("ground_truth.detections.label")
export()

Directory 'yolov5_dataset' already exists; export will be merged with existing files
 100% |█████████████████| 140/140 [1.1s elapsed, 0s remaining, 113.6 samples/s]         
Directory 'yolov5_dataset' already exists; export will be merged with existing files
 100% |███████████████████| 40/40 [327.6ms elapsed, 0s remaining, 122.1 samples/s]      
Directory 'yolov5_dataset' already exists; export will be merged with existing files
 100% |███████████████████| 20/20 [197.1ms elapsed, 0s remaining, 101.5 samples/s]     


In [17]:
model = YOLO("yolo11n.pt")

In [7]:
model_test(dataset)

Processing: C:\Users\knigh\fiftyone\quickstart\data\000880.jpg
  Ground truth: {'bird'}
  Predicted: {'bird'}
  Correct: 1/1
---
Processing: C:\Users\knigh\fiftyone\quickstart\data\001430.jpg
  Ground truth: {'bottle', 'cup', 'chair', 'knife', 'fork', 'dining table'}
  Predicted: {'bottle', 'cup', 'knife', 'fork', 'bowl', 'dining table'}
  Correct: 5/6
---
Processing: C:\Users\knigh\fiftyone\quickstart\data\002284.jpg
  Ground truth: {'bird', 'car', 'truck'}
  Predicted: {'car'}
  Correct: 1/3
---
Processing: C:\Users\knigh\fiftyone\quickstart\data\004548.jpg
  Ground truth: {'broccoli', 'dining table', 'bowl'}
  Predicted: {'broccoli'}
  Correct: 1/3
---
Processing: C:\Users\knigh\fiftyone\quickstart\data\003148.jpg
  Ground truth: {'zebra'}
  Predicted: {'zebra'}
  Correct: 1/1
---
Processing: C:\Users\knigh\fiftyone\quickstart\data\004939.jpg
  Ground truth: {'surfboard', 'person'}
  Predicted: {'person'}
  Correct: 1/2
---
Processing: C:\Users\knigh\fiftyone\quickstart\data\002640.

In [8]:
label = 'sandw_plus'
results = model.predict(f"{label}.jpg", conf=0.5)

In [9]:
annotated_image = show_image(results)

In [10]:
save_image(label, annotated_image)


Annotated image saved as 'sandw_plus_annotated_best.jpg'


In [18]:
video_label = 'sample_video'
process_video(f"{video_label}.mp4", f"{video_label}_output.mp4")

Processing sample_video.mp4...
Video FPS: 30.005645434105602
Press 'q' to quit
Video processing complete. Processed 613 frames in 20.43 seconds.
